# Training-seed robustness experiment

Retrains every subject's Isolation Forest and LSTM Autoencoder under several training seeds and re-evaluates them at 18 sampling resolutions x 10 phase-jitter seeds (51 subjects).

**How to run:** add `DSL-StrongPasswordData.csv` as an input, then use **Save Version -> Save & Run All (Commit)**. It takes roughly 3 hours on a CPU session and keeps running with the browser closed.

**What to bring back:** `training_seeds_eer_by_subject.csv` from the Output tab.

## 1. Settings
Leave these as they are for the real run.

In [ ]:
TRAIN_SEEDS = [42, 1, 2, 3, 4]   # model training seeds
N_JITTER_SEEDS = 10              # phase-jitter seeds per downsampled resolution
N_SUBJECTS = None                # None = all 51 subjects

## 2. Project source files
These cells write the project's code into `src/`.

In [ ]:
import os
for d in ['src', 'data/raw', 'results', 'trial']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/eer.py
"""
Step 3: EER (Equal Error Rate) calculation utility.

EER is the point on the DET curve where FAR (False Acceptance Rate) equals
FRR (False Rejection Rate). scikit-learn has no built-in EER function, so
this implements it from raw anomaly scores.

Convention used here: HIGHER score = more anomalous (more likely an impostor).
If your model outputs "normality" scores instead (higher = more genuine),
negate them before calling these functions, or pass invert_scores=True.
"""
import numpy as np
from scipy.optimize import brentq
from scipy.interpolate import interp1d


def far_frr_at_thresholds(genuine_scores: np.ndarray, impostor_scores: np.ndarray,
                           thresholds: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    A sample is REJECTED (flagged as impostor) if score > threshold.
    FAR = fraction of impostor scores that fall BELOW threshold (wrongly accepted)
    FRR = fraction of genuine scores that fall ABOVE threshold (wrongly rejected)
    """
    genuine_scores = np.asarray(genuine_scores)
    impostor_scores = np.asarray(impostor_scores)

    far = np.array([(impostor_scores <= t).mean() for t in thresholds])
    frr = np.array([(genuine_scores > t).mean() for t in thresholds])
    return far, frr


def compute_eer(genuine_scores: np.ndarray, impostor_scores: np.ndarray,
                 invert_scores: bool = False, n_thresholds: int = 2000) -> dict:
    """
    Returns a dict with:
      eer: the equal error rate (0-1)
      threshold: the score threshold at which FAR == FRR
      far_curve, frr_curve, thresholds: full curves, useful for plotting DET curve
    """
    genuine_scores = np.asarray(genuine_scores, dtype=float)
    impostor_scores = np.asarray(impostor_scores, dtype=float)

    if invert_scores:
        genuine_scores = -genuine_scores
        impostor_scores = -impostor_scores

    lo = min(genuine_scores.min(), impostor_scores.min())
    hi = max(genuine_scores.max(), impostor_scores.max())
    thresholds = np.linspace(lo, hi, n_thresholds)

    far, frr = far_frr_at_thresholds(genuine_scores, impostor_scores, thresholds)

    # far is increasing in threshold, frr is decreasing (roughly) -> find crossing
    diff = far - frr
    # look for a sign change
    sign_changes = np.where(np.diff(np.sign(diff)))[0]

    if len(sign_changes) == 0:
        # no crossing found in range (degenerate case, e.g. perfect separation)
        # fall back to the threshold minimizing |far - frr|
        idx = np.argmin(np.abs(diff))
        eer = (far[idx] + frr[idx]) / 2
        eer_threshold = thresholds[idx]
    else:
        i = sign_changes[0]
        # linear interpolation between thresholds[i] and thresholds[i+1]
        f = interp1d([thresholds[i], thresholds[i + 1]], [diff[i], diff[i + 1]])
        eer_threshold = brentq(f, thresholds[i], thresholds[i + 1])
        far_i = np.interp(eer_threshold, thresholds, far)
        frr_i = np.interp(eer_threshold, thresholds, frr)
        eer = (far_i + frr_i) / 2

    return {
        "eer": float(eer),
        "threshold": float(eer_threshold),
        "far_curve": far,
        "frr_curve": frr,
        "thresholds": thresholds,
    }


if __name__ == "__main__":
    from scipy.stats import norm

    # Self-test with synthetic scores where we KNOW the answer.
    # Two unit-variance normals whose means are d apart cross at the midpoint,
    # so the analytical EER is Phi(-d/2). With 50,000 samples per class the
    # sampling error of the estimate is ~0.2 points, hence the 0.5pt tolerance.
    rng = np.random.default_rng(0)
    n = 50_000
    tolerance = 0.005

    for d in [1.0, 2.0, 3.0]:
        analytical = norm.cdf(-d / 2)
        result = compute_eer(rng.normal(0, 1, n), rng.normal(d, 1, n))
        print(f"Test (means {d:.0f} sigma apart): EER = {result['eer']*100:.2f}%  "
              f"analytical = {analytical*100:.2f}%  threshold = {result['threshold']:.3f}")
        assert abs(result["eer"] - analytical) < tolerance, "EER does not match analytical value"
        assert abs(result["threshold"] - d / 2) < 0.05, "EER threshold is not at the midpoint"

    # Identical distributions -> EER must be 50%
    result2 = compute_eer(rng.normal(0, 1, n), rng.normal(0, 1, n))
    print(f"Test (identical distributions): EER = {result2['eer']*100:.2f}% (analytical 50.00%)")
    assert abs(result2["eer"] - 0.5) < 2 * tolerance

    # Perfectly separated -> EER must be 0%
    result3 = compute_eer(rng.normal(0, 0.1, 1000), rng.normal(10, 0.1, 1000))
    print(f"Test (perfectly separated): EER = {result3['eer']*100:.2f}% (analytical 0.00%)")
    assert result3["eer"] < 0.001

    # invert_scores: "normality" scores (higher = more genuine) give the same EER
    g, i = rng.normal(0, 1, n), rng.normal(2, 1, n)
    assert abs(compute_eer(-g, -i, invert_scores=True)["eer"] - compute_eer(g, i)["eer"]) < 1e-12

    print("All EER self-tests passed.")

In [ ]:
%%writefile src/downsample.py
"""
Step 2: Down-sampling simulation protocol.

Method
------
The CMU dataset gives relative timing features (H = hold time, DD =
down-down latency, UD = up-down latency) per keystroke pair, not raw
timestamps. To simulate what a *coarser-resolution polling system*
would actually observe, we:

  1. Reconstruct absolute press/release timestamps for each of the 11
     keys in ".tie5Roanl" from the relative H/DD features
     (press_1 = 0; press_{i+1} = press_i + DD_i; release_i = press_i + H_i).
  2. Apply a random per-rep phase offset (the "sliding" part of the
     sliding-window protocol) representing the arbitrary alignment of a
     hardware poller's clock relative to the true keystroke event.
  3. Quantize every timestamp to the nearest multiple of the target
     interval (20 ms or 30 ms), which is what a poller sampling at that
     rate would actually report.
  4. Recompute H, DD, UD from the quantized timestamps.

This produces genuine quantization jitter and interpolation artefacts,
rather than naive point-dropping, which is what your methodology
section commits to.

Two modelling choices, and why
------------------------------
Nearest-tick rounding. A real poller reports an event at the NEXT tick
(a ceiling), not the nearest one. Because the phase offset is uniform over
one full poll interval, the two are equivalent for everything measured
here: ceiling is nearest-tick rounding with all timestamps shifted by
dt/2, and a constant shift is absorbed by the uniform phase. H, DD and UD
are differences of timestamps, so their distribution is identical under
either rule.

Floor of dt/2 on collapsed intervals. When a key's press and release (or
two consecutive presses) land on the same tick, the observed interval is
0. The true interval is then known only to be shorter than one poll
interval, so it is replaced by the midpoint dt/2 rather than 0; this
avoids physically impossible zero-length holds. It affects only intervals
shorter than dt, so it is rare at 20-30ms and increasingly common at the
coarse exploratory resolutions (holds are typically ~100ms).
"""
import numpy as np
import pandas as pd

KEYS = ["period", "t", "i", "e", "five", "Shift.r", "o", "a", "n", "l", "Return"]


def _h_cols():
    return [f"H.{k}" for k in KEYS]


def _dd_cols():
    return [f"DD.{KEYS[i]}.{KEYS[i+1]}" for i in range(len(KEYS) - 1)]


def _ud_cols():
    return [f"UD.{KEYS[i]}.{KEYS[i+1]}" for i in range(len(KEYS) - 1)]


def _reconstruct_timestamps(row: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    """Return (press_times, release_times) arrays, length 11, seconds, press_1=0."""
    h = row[_h_cols()].to_numpy(dtype=float)
    dd = row[_dd_cols()].to_numpy(dtype=float)

    press = np.zeros(len(KEYS))
    for i in range(1, len(KEYS)):
        press[i] = press[i - 1] + dd[i - 1]
    release = press + h
    return press, release


def _quantize(timestamps: np.ndarray, dt: float, phase_offset: float) -> np.ndarray:
    """Simulate a poller sampling every dt seconds, with a random phase offset,
    by snapping each timestamp to the nearest poll tick."""
    shifted = timestamps + phase_offset
    quantized = np.round(shifted / dt) * dt - phase_offset
    return quantized


def downsample_row(row: pd.Series, dt: float, rng: np.random.Generator) -> pd.Series:
    """Return a new row with H/DD/UD recomputed as if sampled at interval dt (seconds).
    dt=None (or <= the ~1ms baseline) returns the row unchanged (control group)."""
    if dt is None or dt <= 0.001:
        return row.copy()

    press, release = _reconstruct_timestamps(row)

    # random phase offset per rep: this is the "sliding" in sliding-window
    phase_offset = rng.uniform(0, dt)

    q_press = _quantize(press, dt, phase_offset)
    q_release = _quantize(release, dt, phase_offset)

    # Guard against degenerate quantization collapsing press==release (H=0)
    # or press_i+1 == press_i (DD=0): floor at half a poll tick.
    h_new = np.maximum(q_release - q_press, dt / 2)
    dd_new = np.maximum(np.diff(q_press), dt / 2)
    ud_new = dd_new - h_new[:-1]

    new_row = row.copy()
    for i, k in enumerate(KEYS):
        new_row[f"H.{k}"] = h_new[i]
    for i in range(len(KEYS) - 1):
        new_row[_dd_cols()[i]] = dd_new[i]
        new_row[_ud_cols()[i]] = ud_new[i]
    return new_row


def downsample_dataframe(df: pd.DataFrame, dt: float, seed: int = 0) -> pd.DataFrame:
    """Vectorised equivalent of applying downsample_row to every row in order
    (bit-identical output for the same seed, ~500x faster)."""
    if dt is None or dt <= 0.001:
        return df.copy()

    rng = np.random.default_rng(seed)
    h = df[_h_cols()].to_numpy(dtype=float)
    dd = df[_dd_cols()].to_numpy(dtype=float)

    press = np.concatenate([np.zeros((len(df), 1)), np.cumsum(dd, axis=1)], axis=1)
    release = press + h

    # one phase offset per rep, drawn in row order (same stream as downsample_row)
    phase_offset = rng.uniform(0, dt, size=len(df))[:, None]
    q_press = _quantize(press, dt, phase_offset)
    q_release = _quantize(release, dt, phase_offset)

    h_new = np.maximum(q_release - q_press, dt / 2)
    dd_new = np.maximum(np.diff(q_press, axis=1), dt / 2)

    out = df.copy()
    out[_h_cols()] = h_new
    out[_dd_cols()] = dd_new
    out[_ud_cols()] = dd_new - h_new[:, :-1]
    return out


def build_experimental_groups(df: pd.DataFrame, seed: int = 0) -> dict[str, pd.DataFrame]:
    """Table 1 from the proposal: control (baseline), 20ms, 30ms."""
    return {
        "control_1ms": df.copy(),
        "test_20ms": downsample_dataframe(df, dt=0.020, seed=seed),
        "test_30ms": downsample_dataframe(df, dt=0.030, seed=seed + 1),
    }


if __name__ == "__main__":
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    df = pd.read_csv("data/raw/DSL-StrongPasswordData.csv")
    groups = build_experimental_groups(df)

    for name, g in groups.items():
        g.to_csv(f"data/processed/{name}.csv", index=False)
        print(f"{name}: wrote {len(g)} rows")

    # Validation plot: one subject's first rep, control vs 20ms vs 30ms,
    # for a single feature (H.t) across many reps to see distribution shift
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    subj = df["subject"].iloc[0]
    for ax, (name, g) in zip(axes, groups.items()):
        vals = g[g["subject"] == subj]["H.t"]
        ax.hist(vals, bins=20)
        ax.set_title(name)
        ax.set_xlabel("H.t (seconds)")
    axes[0].set_ylabel("count")
    fig.suptitle(f"Subject {subj}: H.t distribution across sampling groups")
    fig.tight_layout()
    fig.savefig("results/downsample_validation_H_t.png", dpi=120)
    print("Saved results/downsample_validation_H_t.png")

    # Numeric check: mean absolute distortion vs baseline, should grow with dt
    for name, g in groups.items():
        if name == "control_1ms":
            continue
        diff = (g["H.t"] - df["H.t"]).abs().mean()
        print(f"{name}: mean |H.t - baseline| = {diff*1000:.2f} ms")

In [ ]:
%%writefile src/subject_models.py
"""
Shared protocol + model code for one subject (Option A protocol).

Everything the experiment scripts have in common lives here, so a change to
the protocol or to a model is made once:

  Protocol   training_reps()  - a subject's control_1ms training reps (first 200)
             split_subject()  - genuine test reps (last 200) + impostor reps
                                (first 5 of every other subject) from one group
  Models     fit_isolation_forest(), fit_lstm_autoencoder()
             fit_subject_models() - both, trained ONCE on control_1ms
  Scoring    score_samples()  - Isolation Forest, LSTM Autoencoder and Hybrid
                                anomaly scores (higher = more anomalous) for
                                any rows, at any sampling resolution

LSTM sequence framing: each rep of ".tie5Roanl" involves 11 keys. The 31 flat
H/DD/UD columns become a sequence of 11 timesteps, 3 features each:
[H_i, DD_in_i, UD_in_i].
"""
import os
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.dirname(__file__))
from downsample import KEYS

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

FEATURE_PREFIXES = ("H.", "DD.", "UD.")
GROUP_FILES = {
    "control_1ms": "data/processed/control_1ms.csv",
    "test_20ms": "data/processed/test_20ms.csv",
    "test_30ms": "data/processed/test_30ms.csv",
}
MODEL_NAMES = ["Isolation Forest", "LSTM Autoencoder", "Hybrid"]

N_TRAIN_REPS = 200      # sessions 1-4: training; sessions 5-8: genuine test
N_IMPOSTOR_REPS = 5     # first 5 reps of every other subject
N_TIMESTEPS = len(KEYS)
N_FEATURES = 3


# ---------- protocol ----------

def load_groups() -> dict:
    """group name -> full dataframe (all subjects, one sampling rate)."""
    return {name: pd.read_csv(path) for name, path in GROUP_FILES.items()}


def get_feature_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c.startswith(FEATURE_PREFIXES)]


def training_reps(control_df: pd.DataFrame, subject: str) -> pd.DataFrame:
    """The subject's training data: control_1ms ONLY, sessions 1-4."""
    subj_df = (control_df[control_df["subject"] == subject]
               .sort_values(["sessionIndex", "rep"]))
    assert len(subj_df) == 400, f"{subject} has {len(subj_df)} reps"
    return subj_df.iloc[:N_TRAIN_REPS]


def split_subject(df: pd.DataFrame, subject: str) -> tuple:
    """Return (genuine_test_df, impostor_df) for one subject from one group's
    dataframe, i.e. at THAT group's sampling resolution."""
    subj_df = df[df["subject"] == subject].sort_values(["sessionIndex", "rep"])
    genuine_test_df = subj_df.iloc[N_TRAIN_REPS:]

    others = df[df["subject"] != subject]
    impostor_df = (others.sort_values(["subject", "sessionIndex", "rep"])
                         .groupby("subject").head(N_IMPOSTOR_REPS))
    return genuine_test_df, impostor_df


# ---------- LSTM Autoencoder building blocks ----------

def rows_to_sequences(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    seq = np.zeros((n, N_TIMESTEPS, N_FEATURES), dtype=float)
    for i, key in enumerate(KEYS):
        seq[:, i, 0] = df[f"H.{key}"].to_numpy()
        if i > 0:
            prev = KEYS[i - 1]
            seq[:, i, 1] = df[f"DD.{prev}.{key}"].to_numpy()
            seq[:, i, 2] = df[f"UD.{prev}.{key}"].to_numpy()
    return seq


def build_autoencoder(latent_dim: int = 16):
    from tensorflow.keras import layers, models

    inputs = layers.Input(shape=(N_TIMESTEPS, N_FEATURES))
    encoded = layers.LSTM(latent_dim, activation="tanh")(inputs)
    repeated = layers.RepeatVector(N_TIMESTEPS)(encoded)
    decoded = layers.LSTM(latent_dim, activation="tanh", return_sequences=True)(repeated)
    outputs = layers.TimeDistributed(layers.Dense(N_FEATURES))(decoded)
    model = models.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def reconstruction_error(model, X: np.ndarray) -> np.ndarray:
    recon = model.predict(X, verbose=0)
    return np.mean(np.square(X - recon), axis=(1, 2))


def scale_sequences(scaler: StandardScaler, df: pd.DataFrame) -> np.ndarray:
    """Rows -> scaled (n, 11, 3) sequences, using a scaler fitted on training data."""
    X = rows_to_sequences(df)
    n = X.shape[0]
    return scaler.transform(X.reshape(n, -1)).reshape(n, N_TIMESTEPS, N_FEATURES)


# ---------- training (once per subject) ----------

def fit_isolation_forest(train_df: pd.DataFrame, feature_cols: list, seed: int = 42) -> tuple:
    """Returns (model, train_scores)."""
    X_train = train_df[feature_cols].to_numpy()
    model = IsolationForest(n_estimators=100, contamination="auto", random_state=seed)
    model.fit(X_train)
    return model, -model.score_samples(X_train)


def fit_lstm_autoencoder(train_df: pd.DataFrame, seed: int = 42, epochs: int = 60,
                          verbose_fit: int = 0) -> tuple:
    """Returns (model, scaler, train_scores)."""
    import tensorflow as tf

    X_train_raw = rows_to_sequences(train_df)
    scaler = StandardScaler()
    scaler.fit(X_train_raw.reshape(len(X_train_raw), -1))
    X_train = scale_sequences(scaler, train_df)

    tf.keras.utils.set_random_seed(seed)
    model = build_autoencoder()
    early_stop = tf.keras.callbacks.EarlyStopping(monitor="loss", patience=5,
                                                    restore_best_weights=True)
    model.fit(X_train, X_train, epochs=epochs, batch_size=16, verbose=verbose_fit,
              callbacks=[early_stop])
    return model, scaler, reconstruction_error(model, X_train)


@dataclass
class SubjectModels:
    feature_cols: list
    if_model: IsolationForest
    lstm_model: object
    scaler: StandardScaler
    if_train_scores: np.ndarray
    lstm_train_scores: np.ndarray


def fit_subject_models(train_df: pd.DataFrame, feature_cols: list,
                        seed: int = 42, epochs: int = 60) -> SubjectModels:
    """train_df: the subject's control_1ms training reps (see training_reps)."""
    if_model, if_train_scores = fit_isolation_forest(train_df, feature_cols, seed)
    lstm_model, scaler, lstm_train_scores = fit_lstm_autoencoder(train_df, seed, epochs)
    return SubjectModels(feature_cols, if_model, lstm_model, scaler,
                         if_train_scores, lstm_train_scores)


# ---------- scoring ----------

def zscore(train_scores: np.ndarray, scores: np.ndarray) -> np.ndarray:
    """Normalise scores by the subject's own TRAINING-score distribution."""
    mean, std = train_scores.mean(), train_scores.std()
    std = std if std > 1e-8 else 1e-8
    return (scores - mean) / std


def fuse_scores(models: SubjectModels, if_scores: np.ndarray, lstm_scores: np.ndarray,
                 alpha: float = 0.10) -> np.ndarray:
    """Hybrid score. alpha = Isolation Forest weight (0.10 = 10% IF / 90% LSTM,
    Shaheen & Alomari's formula)."""
    return (alpha * zscore(models.if_train_scores, if_scores)
            + (1 - alpha) * zscore(models.lstm_train_scores, lstm_scores))


def score_samples(models: SubjectModels, df: pd.DataFrame, alpha: float = 0.10) -> dict:
    """Score every row of df. Returns model name -> array of anomaly scores."""
    if_scores = -models.if_model.score_samples(df[models.feature_cols].to_numpy())
    lstm_scores = reconstruction_error(models.lstm_model, scale_sequences(models.scaler, df))
    return {
        "Isolation Forest": if_scores,
        "LSTM Autoencoder": lstm_scores,
        "Hybrid": fuse_scores(models, if_scores, lstm_scores, alpha),
    }

In [ ]:
%%writefile src/run_multiseed_experiment.py
"""
Multi-seed robustness run: the same Option A protocol as the other scripts,
but every downsampled resolution is regenerated with several independent
phase-jitter seeds instead of one.

Why this exists
---------------
The downsampling protocol draws a random phase offset per repetition, so a
single run at (say) 20ms is one random draw of the test data. Two single-seed
runs of the same resolution differ by ~0.2-0.3 EER points, which is the same
size as some of the effects being tested (e.g. H2). Repeating each resolution
over several jitter seeds and averaging per subject removes that noise, and
the per-subject output is what hypothesis_verdicts.py runs its paired
significance tests on.

Training is unchanged: ONE Isolation Forest + ONE LSTM Autoencoder per
subject, trained once on control_1ms (first 200 reps). Only the test data is
regenerated per (resolution, jitter seed), so extra seeds add scoring time
only, not training time.

Run: python3 src/run_multiseed_experiment.py              # 10 jitter seeds
     python3 src/run_multiseed_experiment.py --n-subjects 3 --n-seeds 2   # smoke test
"""
import os
import sys
import argparse
import numpy as np
import pandas as pd

sys.path.append(os.path.dirname(__file__))
from eer import compute_eer
from downsample import downsample_dataframe
from subject_models import (N_TRAIN_REPS, N_IMPOSTOR_REPS, get_feature_columns,
                            fit_subject_models, score_samples)

RAW_CSV = "data/raw/DSL-StrongPasswordData.csv"
OUT_CSV = "results/multiseed_eer_by_subject.csv"

# Same resolutions as breaking_point_analysis.py (1 = undownsampled control)
RESOLUTIONS_MS = [1, 20, 30, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 150, 200, 300, 500]


def build_test_conditions(raw_df: pd.DataFrame, n_seeds: int,
                           resolutions_ms: list = RESOLUTIONS_MS) -> pd.DataFrame:
    """Return the test rows (genuine + impostor pool) of every subject under
    every (resolution, jitter seed) condition, stacked into one dataframe with
    added columns resolution_ms, jitter_seed and role ('genuine'/'impostor')."""
    rep_position = raw_df.groupby("subject").cumcount()
    role = pd.Series(np.where(rep_position >= N_TRAIN_REPS, "genuine",
                              np.where(rep_position < N_IMPOSTOR_REPS, "impostor", "")),
                     index=raw_df.index)
    keep = role != ""

    parts = []
    for res_ms in resolutions_ms:
        # the 1ms control has no jitter, so one copy is enough
        seeds = [0] if res_ms == 1 else range(n_seeds)
        for jitter_seed in seeds:
            df = downsample_dataframe(raw_df, dt=res_ms / 1000.0,
                                       seed=jitter_seed * 1000 + res_ms)
            part = df[keep].copy()
            part["role"] = role[keep]
            part["resolution_ms"] = res_ms
            part["jitter_seed"] = jitter_seed
            parts.append(part)
    return pd.concat(parts, ignore_index=True)


def evaluate_subject(raw_df: pd.DataFrame, test_df: pd.DataFrame, subject: str,
                      feature_cols: list, alpha: float = 0.10, seed: int = 42) -> list:
    train_df = raw_df[raw_df["subject"] == subject].iloc[:N_TRAIN_REPS]  # control_1ms ONLY
    models = fit_subject_models(train_df, feature_cols, seed=seed)

    is_genuine = (test_df["subject"] == subject) & (test_df["role"] == "genuine")
    is_impostor = (test_df["subject"] != subject) & (test_df["role"] == "impostor")
    subj_test = test_df[is_genuine | is_impostor]

    # one scoring pass over every condition at once
    scores = score_samples(models, subj_test, alpha=alpha)
    genuine = (subj_test["role"] == "genuine").to_numpy()
    conditions = subj_test.groupby(["resolution_ms", "jitter_seed"], sort=True).indices

    rows = []
    for (res_ms, jitter_seed), idx in conditions.items():
        for model_name, model_scores in scores.items():
            s = model_scores[idx]
            result = compute_eer(s[genuine[idx]], s[~genuine[idx]])
            rows.append({"subject": subject, "resolution_ms": res_ms,
                         "jitter_seed": jitter_seed, "model": model_name,
                         "eer": result["eer"]})
    return rows


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-subjects", type=int, default=None)
    parser.add_argument("--n-seeds", type=int, default=10,
                         help="Independent phase-jitter seeds per downsampled resolution (default 10)")
    parser.add_argument("--alpha", type=float, default=0.10)
    parser.add_argument("--out", type=str, default=OUT_CSV)
    args = parser.parse_args()

    print("=" * 70)
    print(f"MULTI-SEED RUN: {len(RESOLUTIONS_MS)} resolutions x {args.n_seeds} jitter seeds "
          f"(alpha={args.alpha}, Option A protocol)")
    print("=" * 70)

    raw_df = (pd.read_csv(RAW_CSV)
                .sort_values(["subject", "sessionIndex", "rep"]).reset_index(drop=True))
    feature_cols = get_feature_columns(raw_df)

    print("Generating downsampled test data for every (resolution, seed)...")
    test_df = build_test_conditions(raw_df, args.n_seeds)
    print(f"  {test_df.groupby(['resolution_ms', 'jitter_seed']).ngroups} conditions, "
          f"{len(test_df)} test rows")

    subjects = sorted(raw_df["subject"].unique())
    if args.n_subjects:
        subjects = subjects[:args.n_subjects]

    all_rows = []
    for i, subj in enumerate(subjects):
        all_rows.extend(evaluate_subject(raw_df, test_df, subj, feature_cols, alpha=args.alpha))
        # rewrite after every subject so a crash doesn't lose the run so far
        pd.DataFrame(all_rows).to_csv(args.out, index=False)
        print(f"  {subj} ({i+1}/{len(subjects)}) done", flush=True)

    detail_df = pd.DataFrame(all_rows)
    print(f"\nSaved {args.out}")

    summary = (detail_df.groupby(["resolution_ms", "model"])["eer"].mean()
                        .unstack("model").mul(100).round(2))
    print("\nMean EER (%) by resolution and model, averaged over subjects and jitter seeds:")
    print(summary.to_string())

In [ ]:
%%writefile src/run_training_seeds.py
"""
Training-seed robustness run: repeats the whole multi-seed experiment with
several different MODEL TRAINING seeds.

Why this exists
---------------
run_multiseed_experiment.py varies the phase-jitter seed of the TEST data but
trains each subject's models once (seed 42). Training is itself random: the
LSTM Autoencoder starts from random weights and the Isolation Forest builds
random trees. This script retrains every subject's models under several
training seeds, so the hypothesis tests can average over training randomness
as well, and so the spread between training seeds can be reported.

For every training seed it runs the same protocol as
run_multiseed_experiment.py: 18 resolutions x 10 phase-jitter seeds, 51
subjects. Output has one extra column, train_seed.

Designed to be interrupted: results are saved after every subject, and a
restart skips whatever is already on disk.

Run: python3 src/run_training_seeds.py                       # 5 training seeds
     python3 src/run_training_seeds.py --train-seeds 42 1    # choose seeds
     python3 src/run_training_seeds.py --n-subjects 3 --n-seeds 2 --train-seeds 42   # smoke test
"""
import os
import sys
import time
import argparse
import pandas as pd

sys.path.append(os.path.dirname(__file__))
from run_multiseed_experiment import RAW_CSV, RESOLUTIONS_MS, build_test_conditions, evaluate_subject
from subject_models import get_feature_columns

TRAIN_SEEDS = [42, 1, 2, 3, 4]   # 42 is the seed used everywhere else in the project
OUT_DIR = "results/training_seeds"
OUT_CSV = "results/training_seeds_eer_by_subject.csv"
CORE_MS = [1, 20, 30]


def run_one_seed(raw_df: pd.DataFrame, test_df: pd.DataFrame, subjects: list,
                  feature_cols: list, train_seed: int, alpha: float, out_dir: str) -> pd.DataFrame:
    """Evaluate every subject under one training seed, resuming from disk."""
    path = os.path.join(out_dir, f"train_seed_{train_seed}.csv")
    done_df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    done = set(done_df["subject"]) if len(done_df) else set()
    rows = done_df.to_dict("records")

    for i, subj in enumerate(subjects):
        if subj in done:
            continue
        t0 = time.time()
        subj_rows = evaluate_subject(raw_df, test_df, subj, feature_cols, alpha=alpha, seed=train_seed)
        rows.extend({"train_seed": train_seed, **r} for r in subj_rows)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"  train seed {train_seed}: {subj} ({i+1}/{len(subjects)}) done "
              f"in {time.time() - t0:.0f}s", flush=True)

    seed_df = pd.DataFrame(rows)
    return seed_df[seed_df["subject"].isin(subjects)]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-seeds", type=int, nargs="+", default=TRAIN_SEEDS)
    parser.add_argument("--n-subjects", type=int, default=None)
    parser.add_argument("--n-seeds", type=int, default=10,
                         help="Phase-jitter seeds per downsampled resolution (default 10)")
    parser.add_argument("--alpha", type=float, default=0.10)
    parser.add_argument("--out-dir", type=str, default=OUT_DIR)
    parser.add_argument("--out", type=str, default=OUT_CSV)
    args = parser.parse_args()

    print("=" * 70)
    print(f"TRAINING-SEED RUN: training seeds {args.train_seeds}, "
          f"{len(RESOLUTIONS_MS)} resolutions x {args.n_seeds} jitter seeds")
    print("=" * 70)

    os.makedirs(args.out_dir, exist_ok=True)
    raw_df = (pd.read_csv(RAW_CSV)
                .sort_values(["subject", "sessionIndex", "rep"]).reset_index(drop=True))
    feature_cols = get_feature_columns(raw_df)
    test_df = build_test_conditions(raw_df, args.n_seeds)

    subjects = sorted(raw_df["subject"].unique())
    if args.n_subjects:
        subjects = subjects[:args.n_subjects]

    start = time.time()
    parts = []
    for train_seed in args.train_seeds:
        parts.append(run_one_seed(raw_df, test_df, subjects, feature_cols,
                                  train_seed, args.alpha, args.out_dir))
        pd.concat(parts, ignore_index=True).to_csv(args.out, index=False)
        print(f"Training seed {train_seed} complete "
              f"({(time.time() - start) / 60:.0f} min elapsed)\n", flush=True)

    detail_df = pd.concat(parts, ignore_index=True)
    print(f"Saved {args.out}  ({len(detail_df)} rows, "
          f"{detail_df['train_seed'].nunique()} training seeds, "
          f"{detail_df['subject'].nunique()} subjects)")

    core = detail_df[detail_df["resolution_ms"].isin(CORE_MS)]
    summary = (core.groupby(["train_seed", "model", "resolution_ms"])["eer"].mean()
                   .mul(100).unstack("resolution_ms").round(2))
    print("\nMean EER (%) at the core resolutions, per training seed:")
    print(summary.to_string())

## 3. Find the dataset

In [ ]:
import glob, shutil
import pandas as pd

NAME = 'DSL-StrongPasswordData.csv'
candidates = (glob.glob(f'/kaggle/input/**/{NAME}', recursive=True)
              + glob.glob(NAME) + glob.glob(f'../data/raw/{NAME}'))
assert candidates, (f'{NAME} not found. In the right-hand panel choose Add Input -> Upload, '
                    'and upload the CSV as a new dataset, then run again.')
shutil.copy(candidates[0], f'data/raw/{NAME}')
df = pd.read_csv(f'data/raw/{NAME}')
print('Using', candidates[0])
print(df.shape, '-', df['subject'].nunique(), 'subjects')
assert df.shape == (20400, 34) and df['subject'].nunique() == 51, 'This is not the expected CMU file'

In [ ]:
import sys, sklearn, numpy, tensorflow as tf
print('python', sys.version.split()[0], '| tensorflow', tf.__version__, '| scikit-learn', sklearn.__version__,
      '| numpy', numpy.__version__, '| pandas', pd.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'none (CPU run)')

## 4. Two-minute trial
Runs 2 subjects with 1 training seed, to confirm everything works here before the long run starts.

In [ ]:
!{sys.executable} src/run_training_seeds.py --n-subjects 2 --n-seeds 2 --train-seeds 42 --out-dir trial --out trial/trial.csv 2>&1 | grep -v -i -E "warning|oneDNN|tensorflow/core|^W0000|^I0000|^E0000"
trial = pd.read_csv('trial/trial.csv')
assert len(trial) == 2 * 35 * 3 and trial['eer'].between(0, 1).all(), 'Trial run failed'
print('Trial OK:', len(trial), 'rows')

## 5. Full run
About 35-45 seconds per subject per training seed. Progress is saved after every subject, so re-running this cell continues where it stopped.

In [ ]:
seeds = ' '.join(str(s) for s in TRAIN_SEEDS)
subjects = f'--n-subjects {N_SUBJECTS}' if N_SUBJECTS else ''
!{sys.executable} src/run_training_seeds.py --train-seeds {seeds} --n-seeds {N_JITTER_SEEDS} {subjects} 2>&1 | grep -v -i -E "warning|oneDNN|tensorflow/core|^W0000|^I0000|^E0000"

## 6. Check and package the result

In [ ]:
result = pd.read_csv('results/training_seeds_eer_by_subject.csv')
n_subjects = N_SUBJECTS or 51
n_conditions = 1 + 17 * N_JITTER_SEEDS          # 1 ms control + 17 downsampled resolutions
expected = len(TRAIN_SEEDS) * n_subjects * n_conditions * 3
print(len(result), 'rows; expected', expected)
assert len(result) == expected, 'Run is incomplete - re-run the Full run cell to continue'
assert result['eer'].notna().all()
shutil.copy('results/training_seeds_eer_by_subject.csv', 'training_seeds_eer_by_subject.csv')
print('\nDONE. Download training_seeds_eer_by_subject.csv from the Output tab.')